# Data aggregation for 2026 matches

In [ ]:
import joblib
import matplotlib.pyplot as plt
import os
import pandas as pd

Load data

In [ ]:
df_team_stats = pd.read_csv("model/team_stats.csv")
df_matches = pd.read_csv("matches/2026_group_matches.csv")

Join team features to matches

In [ ]:
matches = df_matches.merge(
    df_team_stats,
    left_on="team1",
    right_on="team",
    how="left"
)

matches = matches.rename(
    columns={col: f"{col}_team1" for col in df_team_stats.columns if col != "team"}
)
matches = matches.drop(columns=["team"], errors="ignore")


matches = matches.merge(
    df_team_stats,
    left_on="team2",
    right_on="team",
    how="left"
)

matches = matches.rename(
    columns={col: f"{col}_team2" for col in df_team_stats.columns if col != "team"}
)
matches = matches.drop(columns=["team"], errors="ignore")


matches.head()

Handle new comers, who qualified for the first time 

In [ ]:
global_means = df_team_stats.mean(numeric_only=True)

for col in global_means.index:
    matches[f"{col}_team1"] = matches[f"{col}_team1"].fillna(global_means[col])
    matches[f"{col}_team2"] = matches[f"{col}_team2"].fillna(global_means[col])

assert matches.isna().sum().sum() == 0
matches.head()

Compute diff features

In [ ]:
matches["goals_diff"] = matches["total_goals_team1"] - matches["total_goals_team2"]
matches["matches_diff"] = matches["total_matches_team1"] - matches["total_matches_team2"]
matches["avg_goals_diff"] = matches["avg_goals_per_match_team1"] - matches["avg_goals_per_match_team2"]
matches["num_players_with_goals_diff"] = matches["num_players_with_goals_team1"] - matches["num_players_with_goals_team2"]
matches["max_goals_diff"] = matches["max_goals_by_single_player_team1"] - matches["max_goals_by_single_player_team2"]

matches = matches.drop(columns=["team_team1", "team_team2"], errors="ignore")

matches.head()

# Prediction for 2026 matches

Load model

In [ ]:
clf = joblib.load("model/worldcup_model.pkl")
feature_cols = joblib.load("model/feature_columns.pkl")

Predict winning probability for 2026 matches

In [ ]:
X = matches[feature_cols]
matches["pred_prob_win"] = clf.predict_proba(X)[:, 1]

predictions = matches.loc[:, [
    "match_id",
    "team1",
    "team2",
    "date",
    "group",
    "pred_prob_win"
]].copy()

predictions["predicted_winner"] = predictions.apply(
    lambda row: row["team1"] if row["pred_prob_win"] >= 0.5 else row["team2"],
    axis=1
)

predictions["confidence"] = predictions["pred_prob_win"].apply(
    lambda p: p if p >= 0.5 else 1 - p
)

predictions = predictions.sort_values(
    by=["group", "confidence"],
    ascending=[True, False]
)

predictions.head(6*4)

Output predictions

In [ ]:
predictions.to_csv(
    "predictions/predictions_2026_group_matches.csv",
    index=False
)

Output confidence

In [ ]:
predictions["confidence_bucket"] = pd.cut(
    predictions["confidence"],
    bins=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
)

predictions["confidence_bucket"].value_counts().sort_index()